In [32]:
import pandas as pd

books = pd.read_csv("../data/books_with_categories.csv")

In [33]:
from transformers import pipeline

# classifier for Eckman 6: anger, disgust, fear, joy, neutral, sadness, surprise
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None,
                      device = "mps")
classifier("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528684265911579},
  {'label': 'neutral', 'score': 0.005764600355178118},
  {'label': 'anger', 'score': 0.004419781267642975},
  {'label': 'sadness', 'score': 0.002092392183840275},
  {'label': 'disgust', 'score': 0.001611993182450533},
  {'label': 'fear', 'score': 0.0004138521908316761}]]

In [34]:
books["description"][0]

'"By focusing on one of the principal German formations involved in the Somme fighting, the author brings to life this little-known period, from the initial German advance on the Somme in September 1914 through the formation of the front that became so well known two years later. Covers the early fighting around villages such as Serre, Beaumont-Hamel, Thiepval, Ovillers, La Boisselle and Fricourt.'

In [35]:
classifier(books["description"][0])
# not a very accurate classification

[[{'label': 'neutral', 'score': 0.6494960784912109},
  {'label': 'fear', 'score': 0.12319313734769821},
  {'label': 'disgust', 'score': 0.11645492166280746},
  {'label': 'joy', 'score': 0.04671396687626839},
  {'label': 'anger', 'score': 0.025891738012433052},
  {'label': 'sadness', 'score': 0.023008594289422035},
  {'label': 'surprise', 'score': 0.015241499058902264}]]

In [36]:
# classify sentences individually
classifier(books["description"][0].split("."))

[[{'label': 'neutral', 'score': 0.8091233968734741},
  {'label': 'fear', 'score': 0.06156858056783676},
  {'label': 'joy', 'score': 0.04539727419614792},
  {'label': 'disgust', 'score': 0.030620800331234932},
  {'label': 'surprise', 'score': 0.02831571362912655},
  {'label': 'sadness', 'score': 0.014004391618072987},
  {'label': 'anger', 'score': 0.01096977200359106}],
 [{'label': 'fear', 'score': 0.46994033455848694},
  {'label': 'neutral', 'score': 0.20408938825130463},
  {'label': 'anger', 'score': 0.16336946189403534},
  {'label': 'disgust', 'score': 0.06785716861486435},
  {'label': 'sadness', 'score': 0.06362907588481903},
  {'label': 'surprise', 'score': 0.018177220597863197},
  {'label': 'joy', 'score': 0.012937339954078197}],
 [{'label': 'neutral', 'score': 0.5494773983955383},
  {'label': 'sadness', 'score': 0.11169008165597916},
  {'label': 'disgust', 'score': 0.10400652140378952},
  {'label': 'surprise', 'score': 0.07876541465520859},
  {'label': 'anger', 'score': 0.0641335

In [37]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)
sorted(predictions[0], key=lambda x: x["label"])
# take the sentence with the highest probability for each sentiment

[{'label': 'anger', 'score': 0.01096977200359106},
 {'label': 'disgust', 'score': 0.030620800331234932},
 {'label': 'fear', 'score': 0.06156858056783676},
 {'label': 'joy', 'score': 0.04539727419614792},
 {'label': 'neutral', 'score': 0.8091233968734741},
 {'label': 'sadness', 'score': 0.014004391618072987},
 {'label': 'surprise', 'score': 0.02831571362912655}]

In [38]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

# creates a dictionary for each description containing the maximumm probability for each emotion
def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [39]:
# test for the first 10 books
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [40]:
emotion_scores

{'anger': [np.float64(0.16336946189403534),
  np.float64(0.09151171892881393),
  np.float64(0.04582471400499344),
  np.float64(0.40103963017463684),
  np.float64(0.23973725736141205),
  np.float64(0.01081202644854784),
  np.float64(0.06413350999355316),
  np.float64(0.3238573670387268),
  np.float64(0.06413350999355316),
  np.float64(0.5710240006446838)],
 'disgust': [np.float64(0.10400652140378952),
  np.float64(0.6895232796669006),
  np.float64(0.027481360360980034),
  np.float64(0.1933290809392929),
  np.float64(0.4240506887435913),
  np.float64(0.016964370384812355),
  np.float64(0.10400652140378952),
  np.float64(0.7952403426170349),
  np.float64(0.10400652140378952),
  np.float64(0.5840523838996887)],
 'fear': [np.float64(0.46994033455848694),
  np.float64(0.19474509358406067),
  np.float64(0.05880959331989288),
  np.float64(0.7639957070350647),
  np.float64(0.49011075496673584),
  np.float64(0.03879523277282715),
  np.float64(0.05136270076036453),
  np.float64(0.4789266288280487

In [41]:
from tqdm import tqdm
import re

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = re.split(r'[.?!]+', books["description"][i])
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5000/5000 [04:34<00:00, 18.19it/s]


In [42]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [43]:
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.163369,0.104007,0.469940,0.045397,0.809123,0.111690,0.078765,9781804510407
1,0.091512,0.689523,0.194745,0.025042,0.905126,0.024215,0.078201,9781108663229
2,0.045825,0.027481,0.058810,0.138643,0.876246,0.011528,0.087436,9780367418489
3,0.401040,0.193329,0.763996,0.102078,0.900035,0.425671,0.365718,9781800326514
4,0.239737,0.424051,0.490111,0.031420,0.947440,0.061022,0.116784,9781108485852
...,...,...,...,...,...,...,...,...
4995,0.576595,0.387021,0.121204,0.040564,0.662137,0.111690,0.078765,9781952223174
4996,0.064134,0.104007,0.184558,0.040564,0.549477,0.111690,0.641877,9781532147555
4997,0.106707,0.589851,0.071280,0.017722,0.935639,0.051084,0.095972,9789004278783
4998,0.641640,0.960738,0.542036,0.949692,0.849241,0.289466,0.110474,9781784753474


In [44]:
# add the maximum score for each emotion to the book vectors
books = pd.merge(books, emotions_df, on = "isbn13")

In [45]:
books

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9781804510407,Other Side of the Wire,Ralph J. Whitehead,"Germany. Heer;Germany. Heer. Reserve Corps, 14...","""By focusing on one of the principal German fo...",NaN,2022,NaN,NaN,Other Side of the Wire: Volume 2 - the Battle ...,"9781804510407 ""By focusing on one of the princ...",Nonfiction,0.163369,0.104007,0.469940,0.045397,0.809123,0.111690,0.078765
1,9781108663229,African Literature and the CIA,Caroline Davis,Written communication;African literature;CIA;U...,"""During the period of decolonisation in Africa...",NaN,2020,NaN,NaN,African Literature and the CIA: Networks of Au...,"9781108663229 ""During the period of decolonisa...",Nonfiction,0.091512,0.689523,0.194745,0.025042,0.905126,0.024215,0.078201
2,9780367418489,Interior Provocations,Anca I. Lasc;Pratt Institute Staff,Architecture;Interior architecture;Congresses;...,"""Interior Provocations: History, Theory, and P...",NaN,2020,NaN,NaN,Interior Provocations,"9780367418489 ""Interior Provocations: History,...",Nonfiction,0.045825,0.027481,0.058810,0.138643,0.876246,0.011528,0.087436
3,9781800326514,An Unfortunate Christmas Murder,Hannah Hendy,English literature,"‘Tis the season for gold, frankincense and mur...",https://covers.openlibrary.org/b/id/15179740-L...,2022,NaN,NaN,An Unfortunate Christmas Murder,"9781800326514 ‘Tis the season for gold, franki...",Fiction,0.401040,0.193329,0.763996,0.102078,0.900035,0.425671,0.365718
4,9781108485852,Legitimacy of Unseen Actors in International A...,Freya Baetens,Arbitration (international law);Jurisdiction (...,"""'Unseen actors' are vital to the functioning ...",NaN,2019,NaN,496.0,Legitimacy of Unseen Actors in International A...,"9781108485852 ""'Unseen actors' are vital to th...",Nonfiction,0.239737,0.424051,0.490111,0.031420,0.947440,0.061022,0.116784
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,9781952223174,"Why, As a Muslim, I Support Liberty",Mustafa Akyol,Religion;Philosophy;Liberty;Islam;Muslims;Reli...,"Islam, the second largest religion in the worl...",NaN,2021,NaN,192.0,"Why, As a Muslim, I Support Liberty","9781952223174 Islam, the second largest religi...",Nonfiction,0.576595,0.387021,0.121204,0.040564,0.662137,0.111690,0.078765
4996,9781532147555,The Horror at Happy Landings,Robert Lawrence Stine;Kelly Matthews;Nichole M...,NaN,Two Martians unexpectedly land on Earth and ha...,https://covers.openlibrary.org/b/id/14408331-L...,2020,NaN,NaN,The Horror at Happy Landings: Just Beyond Volu...,9781532147555 Two Martians unexpectedly land o...,Fiction,0.064134,0.104007,0.184558,0.040564,0.549477,0.111690,0.641877
4997,9789004278783,The Blinded State,Mitko B. Panov,"Byzantine empire, history;Macedonia, history;H...","""This book is a revisionist account of Samuel'...",NaN,2019,NaN,478.0,The Blinded State,"9789004278783 ""This book is a revisionist acco...",Nonfiction,0.106707,0.589851,0.071280,0.017722,0.935639,0.051084,0.095972
4998,9781784753474,Walls,Hollie Overton,"Abused women;Murder;Fiction;Fiction, suspense;...","""For fans of The Girl on the Train, The Walls ...",NaN,2018,NaN,416.0,Walls,"9781784753474 ""For fans of The Girl on the Tra...",Fiction,0.641640,0.960738,0.542036,0.949692,0.849241,0.289466,0.110474


In [46]:
books.to_csv("../data/books_with_emotions.csv", index = False)